# Lecture 6 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week06/lecture06_exercise.ipynb`

**Rules:**
1. Use `px` first, then customise with `update_traces` / `update_layout`
2. Colour encodes a meaningful category — not decoration
3. Insight title names the specific finding
4. Consider: would a bar chart be clearer? If yes, use the bar chart

---


In [ ]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('../data/global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [ ]:
# Task 1

fossil = df.loc[df['Source_Type'] == 'Fossil'].copy()

color_discrete_mapping = {
    'Coal': '#D62828',
    'Oil': '#1D4ED8',
    'Natural Gas': '#93C5FD',
}

fig = px.treemap(
    fossil,
    path=['Region', 'Country', 'Source'],
    values='TWh',
    color='Source',
    color_discrete_map=color_discrete_mapping,
    labels={'TWh': 'Energy (TWh)'},
    title='Asia is the most fossil-dependent region, with China and Saudi Arabia leading country totals',
    height=750,
    width=1200,
)

fig.update_traces(
    textinfo='label+value',
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    hovertemplate='<b>%{label}</b><br>%{value:,.0f} TWh<extra></extra>',
)

fig.data[0].marker.colors = [
    c if c in color_discrete_mapping.values() else '#E5E7EB'
    for c in fig.data[0].marker.colors
]

fig.update_layout(
    font=dict(family='Arial', size=12),
    margin=dict(l=10, r=10, t=60, b=10),
    paper_bgcolor='white',
)

fig.show()


## Task 2 — Sunburst: tipping behaviour by day and meal time

**What to build:** A sunburst chart using the built-in `tips` dataset showing how **total bill amount** is distributed across day → time → smoker status.

**Requirements:**
- Load tips with `px.data.tips()`
- Aggregate **total bill** (sum of `total_bill`) per group — not count
- Hierarchy: `path=['day', 'time', 'smoker']`
- Colour encodes smoker status with a CVD-safe blue/orange palette
- Grey out parent nodes (day and time level)
- Use `percent parent` for text labels
- Insight title describing where the most spending happens

> 💡 `tips.groupby(['day', 'time', 'smoker'])['total_bill'].sum().reset_index()`


In [ ]:
# Task 2

tips = px.data.tips()
df_sun = tips.groupby(['day', 'time', 'smoker'])['total_bill'].sum().reset_index()

color_discrete_mapping = {
    'No': '#1D4ED8',
    'Yes': '#D62828',
}

fig = px.sunburst(
    df_sun,
    path=['day', 'time', 'smoker'],
    values='total_bill',
    color='smoker',
    color_discrete_map=color_discrete_mapping,
    labels={'total_bill': 'Total bill (USD)'},
    title='Saturday dinner generates the highest spending, led by non-smokers',
    height=700,
    width=700,
)

fig.update_traces(
    textinfo='label+percent parent',
    hovertemplate='<b>%{label}</b><br>Total bill: $%{value:,.2f}<br>%{percentParent:.0%} of %{parent}<extra></extra>',
    insidetextorientation='radial',
)

fig.data[0].marker.colors = [
    c if c in color_discrete_mapping.values() else '#E5E7EB'
    for c in fig.data[0].marker.colors
]

fig.update_layout(
    font=dict(family='Arial', size=12),
    margin=dict(l=10, r=10, t=60, b=10),
    paper_bgcolor='white',
)

fig.show()


## Task 3 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country


In [ ]:
# Task 3 — charts

low_carbon = (
    df.loc[df['Source_Type'] == 'Low-carbon']
    .groupby('Country', as_index=False)['TWh']
    .sum()
    .sort_values('TWh', ascending=False)
)

low_carbon['All'] = 'Low-carbon'

fig_tree = px.treemap(
    low_carbon,
    path=['All', 'Country'],
    values='TWh',
    color='TWh',
    color_continuous_scale='Blues',
    labels={'TWh': 'Energy (TWh)'},
    title='France leads low-carbon energy output, ahead of Norway and Canada',
    height=650,
    width=1000,
)

fig_tree.update_traces(
    textinfo='label+value',
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    hovertemplate='<b>%{label}</b><br>%{value:,.0f} TWh<extra></extra>',
    root_color='white',
)

fig_tree.update_layout(
    font=dict(family='Arial', size=12),
    margin=dict(l=10, r=10, t=60, b=10),
    paper_bgcolor='white',
)

fig_tree.show()

fig_bar = px.bar(
    low_carbon,
    x='TWh',
    y='Country',
    orientation='h',
    color_discrete_sequence=['#D62828'],
    labels={'TWh': 'Low-carbon energy (TWh)', 'Country': ''},
    title='France leads low-carbon energy output among countries',
    height=700,
    width=1000,
)

fig_bar.update_traces(
    texttemplate='%{x:,.0f} TWh',
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>%{x:,.0f} TWh<extra></extra>',
)

fig_bar.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    showlegend=False,
)
fig_bar.update_xaxes(gridcolor='#EEEEEE')
fig_bar.update_yaxes(categoryorder='total ascending')

fig_bar.show()


**Answer:** The horizontal bar chart is clearer here because it makes the country ranking and the size differences easier to compare precisely. The treemap is compact and useful for overview, but the bar chart better shows that France leads low-carbon energy output.
